# Convert the Classification Datasets to PCAP

In [1]:
import os
from pathlib import Path


os.chdir(Path.cwd().parents[0])
print("Now in:", Path.cwd())

dataPathProcessed = str(Path.cwd()) + r"\data\csv" + r"\Processed Files\\"
dataPathProcessedPCAP = str(Path.cwd()) + r"\data\pcap" + r"\Processed Files\\"

Now in: C:\Users\stsax\OneDrive\Studium\9. Semester\Masterarbeit\Repository


In [2]:
import pandas as pd

pkt_df_train = pd.read_parquet(dataPathProcessed + r"classification_dataset_v2_train.parquet")
seq_table_train = pd.read_parquet(dataPathProcessed + r"classification_sequence_table_v2_train.parquet")
stream_idx_train = pd.read_parquet(dataPathProcessed + r"classification_stream_index_v2_train.parquet")

pkt_df_test = pd.read_parquet(dataPathProcessed + r"classification_dataset_v2_test.parquet")
seq_table_test = pd.read_parquet(dataPathProcessed + r"classification_sequence_table_v2_test.parquet")
stream_idx_test = pd.read_parquet(dataPathProcessed + r"classification_stream_index_v2_test.parquet")


In [3]:
from modeling.BleDataset import BLEStreamDataset
import pickle

MaskConfigPath = str(Path.cwd()) + r"\data_masking\mask_configs\\"
picklePath = str(Path.cwd()) + r"\out\pickle_objects\bpe" + "\\"

with open(picklePath + 'Fitted_BPE_State_Dict.pickle', 'rb') as f:
    state_dict = pickle.load(f)

MAX_TOKEN_LENGTH = 1024
MAX_SEQUENCE_LENGTH = 32

dataset_train = BLEStreamDataset(packet_df=pkt_df_train, sequence_table=seq_table_train, stream_index=stream_idx_train,
                           max_sequence_length=MAX_SEQUENCE_LENGTH, max_token_length=MAX_TOKEN_LENGTH,
                           tokenizer_dict=state_dict, mask_config_path=MaskConfigPath, dataset_type='train')

dataset_test = BLEStreamDataset(packet_df=pkt_df_test, sequence_table=seq_table_test, stream_index=stream_idx_test,
                           max_sequence_length=MAX_SEQUENCE_LENGTH, max_token_length=MAX_TOKEN_LENGTH,
                           tokenizer_dict=state_dict, mask_config_path=MaskConfigPath, dataset_type='test')

In [4]:
import torch

train_loader = torch.utils.data.DataLoader(dataset_train, batch_size=50, num_workers=8, pin_memory=True,     persistent_workers=True,     drop_last=False,     prefetch_factor=4,)

validation_loader = torch.utils.data.DataLoader(dataset_test, batch_size=50, num_workers=8, pin_memory=True,     persistent_workers=True,     drop_last=False,     prefetch_factor=4, shuffle=False)

In [5]:
from tqdm import tqdm
from ble import BleStream, Packet

stream = BleStream()
label_list_train = []


for idx in tqdm(range(len(dataset_train))):
    out_dict = dataset_train[idx]
    packet = Packet()
    packet.from_string(out_dict['masked packets'][0][2], parse_mode='tolerant')

    stream.add_packet(packet, update=True)
    label_list_train.append(out_dict['label'])


with open(dataPathProcessedPCAP + "Dataset_Classification_Label_List_Train" + '.pickle', 'wb') as f:
    pickle.dump(label_list_train, f)

stream.to_pcap_file(dataPathProcessedPCAP + "Dataset_Classification_Train" + '.pcap')


100%|██████████| 30000/30000 [1:13:50<00:00,  6.77it/s]
30000it [00:08, 3547.20it/s]


In [6]:
stream = BleStream()
label_list_test = []


for idx in tqdm(range(len(dataset_test))):
    out_dict = dataset_test[idx]
    packet = Packet()
    packet.from_string(out_dict['masked packets'][0][2], parse_mode='tolerant')

    stream.add_packet(packet, update=True)
    label_list_test.append(out_dict['label'])


with open(dataPathProcessedPCAP + "Dataset_Classification_Label_List_Test" + '.pickle', 'wb') as f:
    pickle.dump(label_list_test, f)

stream.to_pcap_file(dataPathProcessedPCAP + "Dataset_Classification_Test" + '.pcap')

100%|██████████| 6000/6000 [16:00<00:00,  6.25it/s]
6000it [00:01, 4491.71it/s]
